In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.svm import SVC

In [2]:
df = pd.read_csv('train.csv')

In [3]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [4]:
df.dtypes

Loan_ID               object
Gender                object
Married               object
Dependents            object
Education             object
Self_Employed         object
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area         object
Loan_Status           object
dtype: object

In [5]:
df = df.drop(columns=['Loan_ID','Married','Dependents'])

In [6]:
df.isnull().sum()

Gender               13
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [7]:
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

In [8]:
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

In [9]:
le_education = LabelEncoder()
df['Education'] = le_education.fit_transform(df['Education'])
le_self_employee = LabelEncoder()
df['Self_Employed'] = le_self_employee.fit_transform(df['Self_Employed'])
le_Gender = LabelEncoder()
df['Gender'] = le_Gender.fit_transform(df['Gender'])
le_loan_status = LabelEncoder()
df['Loan_Status'] = le_loan_status.fit_transform(df['Loan_Status'])

In [10]:
df = pd.get_dummies(df,columns=['Property_Area'],drop_first=True, dtype=int)

In [11]:
X = df.drop(columns =['Loan_Status'])
y = df['Loan_Status']

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('svm', SVC(random_state=42))
])

In [13]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipe, X_train, y_train, cv=5)
print(scores, scores.mean())

[0.84848485 0.80612245 0.82653061 0.78571429 0.80612245] 0.8145949288806431


In [14]:
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': [0.01, 0.1, 1]
}
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', MinMaxScaler()),
                                       ('svm', SVC(random_state=42))]),
             param_grid={'svm__C': [0.1, 1, 10, 100],
                         'svm__gamma': [0.01, 0.1, 1]})

In [15]:
print(grid.best_params_, grid.best_score_)


{'svm__C': 1, 'svm__gamma': 0.1} 0.8145949288806431


In [16]:
print(grid.score(X_test, y_test))

0.7886178861788617


In [17]:
y_pred = grid.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[18 25]
 [ 1 79]]
0.7886178861788617
              precision    recall  f1-score   support

           0       0.95      0.42      0.58        43
           1       0.76      0.99      0.86        80

    accuracy                           0.79       123
   macro avg       0.85      0.70      0.72       123
weighted avg       0.83      0.79      0.76       123



In [18]:
rule_pred = (X_test['Credit_History'] > 0.5).astype(int)
print((y_pred == rule_pred).mean())

1.0


In [19]:
pd.DataFrame(grid.cv_results_)[['params','mean_test_score','rank_test_score']].sort_values('rank_test_score')

,params,mean_test_score,rank_test_score
6,"{'svm__C': 10, 'svm__gamma': 0.01}",0.814595,1
7,"{'svm__C': 10, 'svm__gamma': 0.1}",0.814595,1
4,"{'svm__C': 1, 'svm__gamma': 0.1}",0.814595,1
9,"{'svm__C': 100, 'svm__gamma': 0.01}",0.814595,1
10,"{'svm__C': 100, 'svm__gamma': 0.1}",0.804391,5
8,"{'svm__C': 10, 'svm__gamma': 1}",0.804350,6
5,"{'svm__C': 1, 'svm__gamma': 1}",0.802350,7
11,"{'svm__C': 100, 'svm__gamma': 1}",0.783983,8
2,"{'svm__C': 0.1, 'svm__gamma': 1}",0.782066,9
1,"{'svm__C': 0.1, 'svm__gamma': 0.1}",0.696537,10
